# 01. 상품목록 크롤링

giftco.co.kr 관리자 페이지(`partner_goods_admlist`)를 페이지 단위로 순회하며
입점업체 상품 목록을 수집합니다. 이 노트북의 결과(상품링크)는
`02_crawl_supplier_info.ipynb`의 입력으로 사용됩니다.

- **입력**: 없음 (giftco.co.kr을 직접 크롤링)
- **출력**: `partner_goods_full.xlsx` (전체 결과), `partner_goods_checkpoint.xlsx` (중간 저장, 중단 시 이어받기용)
- **필요 환경변수**: `.env`의 `GIFTCO_SUPPLIER_COOKIE` (giftco.co.kr 로그인 세션 쿠키)
- **실행 방법**: 셀을 순서대로 실행하면 마지막 셀에서 크롤링이 시작됩니다. 중단되었다가 다시 실행해도
  `partner_goods_checkpoint.xlsx`가 있으면 마지막으로 수집된 페이지 다음부터 자동으로 이어받습니다.


In [ ]:
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry

load_dotenv()

# ============================================================
# 0. 설정
# ============================================================
GIFTCO_SUPPLIER_COOKIE = os.environ.get("GIFTCO_SUPPLIER_COOKIE", "").strip()
if not GIFTCO_SUPPLIER_COOKIE:
    raise RuntimeError(
        ".env에 GIFTCO_SUPPLIER_COOKIE가 설정되지 않았습니다. "
        "giftco.co.kr 로그인 후 F12 > Network 탭에서 아무 요청이나 클릭 > "
        "Request Headers의 Cookie 값을 통째로 복사해 .env에 넣어주세요."
    )

BASE_URL = "https://giftco.co.kr/mypage/page.php?code=partner_goods_admlist&page={}"

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
CHECKPOINT_PATH = f"{DATA_DIR}/partner_goods_checkpoint.xlsx"   # 중간 저장 (중단/재개용)
OUTPUT_PATH = f"{DATA_DIR}/partner_goods_full.xlsx"             # 최종 결과

PAGE_START_DEFAULT = 1      # 체크포인트가 없을 때 시작 페이지
PAGE_END = 5000              # 안전 상한선 (실제 마지막 페이지에서 자동으로 멈춤)
PAGE_END = 20             # 테스트
CHECKPOINT_EVERY = 20         # 몇 페이지마다 체크포인트 저장할지
SLEEP_SEC = 0.5               # 요청 사이 대기(초) — 서버 부담/차단 방지용
TIMEOUT = 15

HEADERS = {
    "Cookie": GIFTCO_SUPPLIER_COOKIE,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
}

# 세션 + 재시도 (일시적 오류/차단에 대비)
session = requests.Session()
session.headers.update(HEADERS)
retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))


def save_excel(df, path):
    """문자열을 URL로 자동변환하지 않도록 저장 (대량 링크 저장 시 엑셀 한도 경고 방지)."""
    with pd.ExcelWriter(
        path, engine="xlsxwriter",
        engine_kwargs={"options": {"strings_to_urls": False}},
    ) as writer:
        df.to_excel(writer, index=False)


In [ ]:
# ============================================================
# 1. 페이지 파서
# ============================================================
def clean(s):
    return re.sub(r"\s+", " ", s).strip()


def parse_page(html):
    """관리자 상품목록 페이지 1개에서 상품 레코드 목록을 추출."""
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", id="sodr_list")
    if not table or not table.find("tbody"):
        return []

    trs = table.find("tbody").find_all("tr", recursive=False)
    records, i = [], 0
    while i < len(trs):
        # 윗줄(체크박스 있는 줄)을 찾아 바로 다음 줄과 짝짓기
        if trs[i].find("input", {"name": "chk[]"}) and i + 1 < len(trs):
            top = trs[i].find_all("td", recursive=False)
            bot = trs[i + 1].find_all("td", recursive=False)
            if len(top) >= 12 and len(bot) >= 5:
                hid = top[0].find("input", {"type": "hidden"})
                gs_id = hid["value"] if hid else ""

                a = top[2].find("a")
                # 상품 상세 링크(href) 그대로 가져와 절대경로로 보정
                shop_url = a["href"].strip() if (a and a.has_attr("href")) else ""
                m = re.search(r"index_no=(\d+)", shop_url) if shop_url else None
                index_no = m.group(1) if m else ""
                if shop_url and not shop_url.startswith("http"):
                    shop_url = "https://giftco.co.kr" + (shop_url if shop_url.startswith("/") else "/" + shop_url)
                if not shop_url and index_no:
                    shop_url = f"https://giftco.co.kr/shop/view.php?index_no={index_no}"

                img = top[2].find("img")
                img_url = img["src"] if img else ""

                parts = list(top[1].stripped_strings)
                no = parts[0] if parts else ""
                expose = parts[1] if len(parts) > 1 else ""

                name_cell = top[4]
                badge = name_cell.find("span")
                ban = "Y" if (badge and "퍼가기" in badge.get_text()) else "N"
                if badge:
                    badge.extract()

                records.append({
                    "번호": no,
                    "노출": expose,
                    "상품코드": clean(top[3].get_text()),
                    "업체코드": clean(bot[0].get_text()),
                    "상품명": clean(name_cell.get_text()),
                    "퍼가기금지": ban,
                    "업체명": clean(bot[1].get_text()),
                    "카테고리": clean(bot[2].get_text()),
                    "최초등록일": clean(top[5].get_text()),
                    "최근수정일": clean(bot[3].get_text()),
                    "진열": clean(top[6].get_text()),
                    "과세": clean(bot[4].get_text()),
                    "기본수량": clean(top[7].get_text()),
                    "공급가": clean(top[8].get_text()),
                    "판매가1": clean(top[9].get_text()),
                    "판매가7": clean(top[10].get_text()),
                    "마진율": clean(top[11].get_text()),
                    "gs_id": gs_id,
                    "index_no": index_no,
                    "상품링크": shop_url,
                    "이미지URL": img_url,
                })
            i += 2
        else:
            i += 1
    return records


In [ ]:
# ============================================================
# 2. 크롤링 실행 (체크포인트가 있으면 마지막 페이지 다음부터 자동으로 이어받음)
# ============================================================
def crawl():
    if os.path.exists(CHECKPOINT_PATH):
        prev = pd.read_excel(CHECKPOINT_PATH)
        all_records = prev.to_dict("records")
        start_page = int(prev["page"].max()) + 1 if "page" in prev.columns and len(prev) else PAGE_START_DEFAULT
        print(f"[info] 체크포인트에서 {len(all_records)}건 불러옴 → {start_page}페이지부터 이어서 진행")
    else:
        all_records = []
        start_page = PAGE_START_DEFAULT

    for page in tqdm(range(start_page, PAGE_END)):
        try:
            res = session.get(BASE_URL.format(page), timeout=TIMEOUT)
        except requests.exceptions.RequestException as e:
            print(f"\n{page}페이지 실패({e}) → 5초 후 재시도")
            time.sleep(5)
            try:
                res = session.get(BASE_URL.format(page), timeout=TIMEOUT)
            except requests.exceptions.RequestException:
                print(f"{page}페이지 재시도 실패 → 중단")
                break

        res.encoding = res.apparent_encoding
        recs = parse_page(res.text)
        for r in recs:
            r["page"] = page

        if not recs:
            if "sodr_list" not in res.text:
                print(f"\n⚠️ {page}페이지: 로그인 페이지로 보임 → 세션 만료 가능. GIFTCO_SUPPLIER_COOKIE 갱신 필요")
            else:
                print(f"\n{page}페이지: 마지막 페이지로 보임 → 정상 종료")
            break

        all_records.extend(recs)
        if page % CHECKPOINT_EVERY == 0:
            save_excel(pd.DataFrame(all_records), CHECKPOINT_PATH)
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(all_records).drop_duplicates(subset="gs_id", keep="first")
    save_excel(df, OUTPUT_PATH)
    print(f"전체 {len(df)}건 저장 완료 → {OUTPUT_PATH}")
    return df


df = crawl()
